# 🪙 Counterfactuals

What input do I need to change in order to get a different result? Which input do I need to change in order to produce a specific output?

Counterfactuals are a way to look into find ways of producing different outputs by minimally modifying theinitial input. This is useful when looking for easy ways of achieving specific desired results, answering “what is the smallest change I can make to get the result I want?”

In our case, it might be that we wanted to check what input we needed to make the song popular in Turkey.  
We will be using TrustyAI to test exactly this, and see how much we would need to change.

In [ ]:
%pip -q install numpy pandas onnxruntime torch trustyai jpype1==1.6.0
%pip -q install dotenv

In [ ]:
# import libraries
try:
    import os
    import sys
    import onnxruntime as rt
    import pickle
    from trustyai.model import Model, feature, output
    from trustyai.explainers import CounterfactualExplainer
    import pandas as pd
    import numpy as np
    import torch

    # silence warnings
    import warnings
    warnings.filterwarnings("ignore", category=UserWarning)
except ImportError as e:
    print(f"{e}")

# local imports
parent_dir = os.path.abspath('..')
# the parent_dir could already be there if the kernel was not restarted, and we run this cell again
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

try:
    from libs.utilities import download_from_s3, download_file
except ImportError as e:
    print(f"{e}")

In [ ]:
# data information
data_bucket: str = "jukebox/artifacts"
data_version: int = 1
data_pointer: str = f"{data_bucket}/{data_version}"

# global variables
DATASET_LOCAL_PATH: str = "../dataset.local"
MODELS_LOCAL_PATH: str = "../models.local"
ENVFILE: str = ".env"

# setup proxy settings to access the network (OPTIONAL)
proxies: dict = {
    "http": "",
    "https": "",
}

# data files to download
SONG_PROPERTIES: dict = {
    "file": f"{DATASET_LOCAL_PATH}/parquet/song_properties.parquet",
    "url": 'https://github.com/rhoai-mlops/jukebox/raw/refs/heads/main/99-data_prep/song_properties.parquet',
}

SONG_RANKINGS: dict = {
    "file": f"{DATASET_LOCAL_PATH}/parquet/song_rankings.parquet",
    "url": 'https://github.com/rhoai-mlops/jukebox/raw/refs/heads/main/99-data_prep/song_rankings.parquet',
}

# Download the datasets if they don't exist locally
download_file(SONG_PROPERTIES["url"], SONG_PROPERTIES["file"], proxy=proxies)
download_file(SONG_RANKINGS["url"], SONG_RANKINGS["file"], proxy=proxies)

Let's start by choosing a country we want the song to be popular in.  
We also pick what probability we need to see before we say that there's a good chance that our song will be popular in that country.  

In [ ]:
# Counterfactual Analysis Features
PRED_COUNTRY = "TR"
POPULAR_THRESHOLD = 0.3

We then load our model, as well as our pre-and-post-processing artifacts.  

In [ ]:
# get api tokens to access to s3
if os.path.exists(ENVFILE):
    # load environment using dotenv
    print(f"Loading dotenv environment from {ENVFILE}")
    import dotenv
    dotenv.load_dotenv()

aws_s3_endpoint: str = os.environ.get("AWS_S3_ENDPOINT", "")
access_key: str = os.environ.get("AWS_ACCESS_KEY_ID", "")
secret_key: str = os.environ.get("AWS_SECRET_ACCESS_KEY", "")
aws_bucket: str = os.environ.get("AWS_S3_BUCKET_NAME", "")

# prepare access credentials dictionary for s3 access
access_creds: dict = {
    "aws_access_key_id": access_key,
    "aws_secret_access_key": secret_key,
    "aws_s3_endpoint": aws_s3_endpoint,
    "aws_s3_bucket_name": aws_bucket,
}

# we download training artifacts from the s3 registry
training_artifacts = {
    "training_data": f"{data_pointer}/train_dataset.parquet",
    "test_data": f"{data_pointer}/test_dataset.parquet",
    "scaler": f"{data_pointer}/minmax_scaler.pkl",
    "label_encoder": f"{data_pointer}/label_encoder.pkl",
    "model_checkpoint": "jukebox/onnx/1/country_predictor_model.onnx",
    "merged_data": f"{data_pointer}/merged_dataset_full.parquet",
}

In [ ]:
# download resources from s3
for k in training_artifacts.keys():
    obj_id = training_artifacts.get(k)
    print(f"Getting object: {obj_id}")
    download_from_s3(aws_s3_endpoint,
                     aws_bucket,
                     obj_id,
                     obj_id,
                     access_creds
                    )

In [ ]:
# features...
dataset_features = ['is_explicit', 'duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# load encoder and scaler
with open(training_artifacts.get("scaler"), 'rb') as handle:
    scaler = pickle.load(handle)
    print(scaler)

with open(training_artifacts.get("label_encoder"), 'rb') as handle:
    label_encoder = pickle.load(handle)
    print(label_encoder)

In [ ]:
# 1- Load prediction model from disk
predictor = rt.InferenceSession(training_artifacts.get("model_checkpoint"),
                                providers=rt.get_available_providers())

# 2- get inputs and outputs of the onnx model
onnx_input = predictor.get_inputs()[0]
onnx_output = predictor.get_outputs()[0]
input_name = onnx_input.name
output_name = onnx_output.name

print(f"ONNX Model:\n Input: {input_name}, shape: {onnx_input.shape}\n Output: {output_name}, shape: {onnx_output.shape}")

### Data

Then we pick a song we want to try to make popular in that country.  
We will also process the song properties a bit, such as scaling them, just like what we did when training the model. This is to make sure they have an input that the model understands. 

In [ ]:
song_properties = pd.read_parquet(f"{SONG_PROPERTIES.get("file")}")
favorite_song = song_properties.loc[song_properties["name"]=="Not Like Us"]
favorite_song

In [ ]:
song_properties = favorite_song[dataset_features]
song_properties.T

In [ ]:
# scale features between [-1,1]
scaled_features = scaler.transform(song_properties)[0]

# the scaled feature values for the current song
feature_values = {
    "is_explicit": scaled_features[0],
    "duration_ms": scaled_features[1],
    "danceability": scaled_features[2],
    "energy": scaled_features[3],
    "key": scaled_features[4],
    "loudness": scaled_features[5],
    "mode": scaled_features[6],
    "speechiness": scaled_features[7],
    "acousticness": scaled_features[8],
    "instrumentalness": scaled_features[9],
    "liveness": scaled_features[10],
    "valence": scaled_features[11],
    "tempo": scaled_features[12]
}

# convert input feature vector to a float tensor
feature_df = pd.DataFrame([feature_values])
feature_df.T

We also set what all the output names should be called, this will be the same as the country codes.

In [ ]:
output_names = label_encoder.classes_
output_names

### Counterfactual analysis

Now that we have all of this set up, will set up our counterfactual analysis. We need to set up the TrustyAI SDK to do this.

1. Create a model runner function

Here we need to first create a predict function that will use the Predictive PyTORCH model to output a prediction given a set of input features.
The predict function will take in a dataframe with all of our features and return the prediction for that datapoint.
- Input: Pandas DataFrame with all of our features (shape: (1, 13)).
- Output: Prediction for that datapoint (shape: (1, 73)).

2. Create a prediction function

We need a prediction function that will take in our datapoint and return the likelihood that the song will be popular in the given country.
- Input: The model, the dataframe with all of our features (shape: (1, 13)), the target feature set, the threshold of acceptability
- Output: The likelihood that the song will be popular in the given country (DataFrame, boolean, shape: (1, 1))

3. Create a Trusty AI "Model"

Then we will create a TrustyAI "Model", this just wraps our model and will be used by TrustyAI to iterate on different input values.

4. Define search domain

Finally, we will define TrustyAI "domains" for each of our inputs. This tells TrustyAI what values the input is allowed to be between.

In [ ]:
def run_onnx_model(model, datapoint: pd.DataFrame) -> np.ndarray:
    # get inputs and outputs of the onnx model
    onnx_input = model.get_inputs()[0]
    onnx_output = model.get_outputs()[0]
    input_name = onnx_input.name
    output_name = onnx_output.name

    # convert to torch tensor
    datapoint = torch.tensor(datapoint.values).float()

    # make a prediction using the model
    point = datapoint.reshape(onnx_input.shape)
    y_hat_temp = model.run([output_name], {input_name: point.numpy()})
    
    # return prediction
    return y_hat_temp[0][0]

def make_prediction(model,
                    datapoint: pd.DataFrame,
                    output_features,
                    tgt_feature,
                    threshold=0.5) -> pd.DataFrame:
    """Make a prediction using the model"""
    prediction = run_onnx_model(model, datapoint=datapoint)
    
    # map prediction to value in dictionary format
    prediction = {output_features[i]: prediction[i] for i in range(len(output_features))}
        
    # find countries that cross the acceptance threshold
    if prediction[tgt_feature] >= threshold:
        y_hat = {tgt_feature: True}
    else:
        y_hat = {tgt_feature: False}

    # return DataFrame
    return pd.DataFrame([y_hat])

In [ ]:
# run the model
y_hat_df = make_prediction(predictor,
                            feature_df,
                            output_names,
                            PRED_COUNTRY,
                            POPULAR_THRESHOLD)

# show prediction output
print(y_hat_df)

In [ ]:
# define feature domains for the TrustyAI Explainer to work on
features_domain: tuple = (-1.0, 1.0) # features are scaled between -1 and 1

# create features out of values/domains
features = []
for key in feature_values.keys():
    features.append(feature(f"{key}", "number", value=feature_values.get(key), domain=features_domain))

# have a look
for feature in features:
    print(feature)

In [ ]:
# define the output goal: the feature we want to get by slightly altering the input values
goal = [output(name=PRED_COUNTRY, dtype="bool", value=True)]

In [ ]:
# load the TrustyAI Model
def trustyai_predictor_fn(input_features, model=predictor, output_features=output_names, tgt_feature=PRED_COUNTRY, threshold=POPULAR_THRESHOLD):
    # call the torch model
    return make_prediction(
        model=model, 
        datapoint=pd.DataFrame(input_features),
        output_features=output_names, tgt_feature=tgt_feature, threshold=threshold
    )

# Wrap the predictive model with a TrustyAI Model object
model = Model(trustyai_predictor_fn, output_names=[PRED_COUNTRY])

After we have the model, the domains, and the goal, we can start running through possible inputs to see which one can give us the output we want.  

In [ ]:
STEPS=5000
explainer = CounterfactualExplainer(steps=STEPS)
explanation = explainer.explain(inputs=features,
                                goal=goal,
                                model=model)

In [ ]:
model(explanation.proposed_features_dataframe.to_numpy())

Now that it has finished running, we can see how much we would need to change our original input (remember the song we chose at the start) for the song to become popular in our country.  

In [ ]:
explanation.as_dataframe()

In [ ]:
df = explanation.as_dataframe()
df[df.difference != 0.0]

In [ ]:
if not df[df.difference != 0.0].empty:
    explanation.plot()
else:
    print(f"The country {PRED_COUNTRY} did not reach the probability {POPULAR_THRESHOLD} in {STEPS} steps")